# Phase 05 · Bài 08 — CNN và RNN cho văn bản

**Mục tiêu:** hiểu TextCNN như bộ dò n-gram học được; tự cài convolution 1D + max-pool; cài một bước LSTM từ công thức cổng; định lượng vanishing gradient; biết khi nào kiến trúc tuần tự vẫn thắng transformer.

## 1. Vì sao cần kiến trúc mới

TF-IDF và Word2Vec tạo vector phẳng, bỏ thứ tự từ. Bộ phân loại trên chúng không phân biệt `dog bites man` với `man bites dog`. Thứ tự đôi khi mang tín hiệu, và có hai họ kiến trúc lấp khoảng trống trước khi attention xuất hiện.

**TextCNN.** Áp convolution 1D lên chuỗi embedding. Filter rộng 3 là bộ dò trigram học được: nó trải trên ba từ và cho một điểm. Ghép nhiều độ rộng (2, 3, 4, 5) để bắt mẫu nhiều thang. Max-pool về vector kích thước cố định. Phẳng, song song, nhanh.

**RNN/LSTM/GRU.** Xử lý từng token, giữ hidden state mang thông tin về phía trước. Tuần tự, có bộ nhớ, linh hoạt độ dài. Thống trị mô hình chuỗi 2014–2017, rồi attention đến.

## 2. TextCNN: cơ chế

Cho embedding matrix X có kích thước L × d (L token, d chiều). Một filter W kích thước k × d trượt qua chuỗi; tại vị trí i:

a_i = ReLU( Σ_{j=0}^{k-1} Σ_{c=0}^{d-1} X[i+j, c] · W[j, c] + b )

Kết quả là feature map a có L − k + 1 phần tử. Global max-pool lấy max theo thời gian:

z = max_i a_i

Ghép z của nhiều filter/độ rộng, đưa qua lớp tuyến tính.

Ba tính chất quan trọng:

1. **Filter là n-gram detector.** Trọng số học được quyết định cụm nào kích hoạt. Không ai viết luật "not good"; mô hình tự học nếu cụm đó tương quan với nhãn.
2. **Max-pool bất biến vị trí.** `not good` ở đầu hay giữa review đều kích hoạt cùng một feature. Với phân loại, đây là mong muốn; với trích xuất vị trí, đây là hạn chế.
3. **Song song theo thời gian.** Mọi cửa sổ tính độc lập, nên GPU xử lý cả câu cùng lúc.

Đổi lại, cửa sổ cố định: filter rộng 3 không bao giờ thấy quan hệ giữa token 1 và token 50. Muốn thấy xa hơn phải tăng k, và chi phí tham số tăng theo.

## 3. RNN và vanishing gradient

RNN cập nhật trạng thái ẩn tại mỗi bước:

h_t = tanh(W x_t + U h_{t-1} + b)

`W`, `U`, `b` chia sẻ qua mọi bước. Hidden state tại T là bản tóm tắt toàn bộ tiền tố. Với phân loại, pool trên h_1 … h_T (max, mean hoặc last).

Huấn luyện dùng backpropagation through time: gradient từ bước T phải đi ngược qua T−1 phép nhân với `U`. Nếu bán kính phổ của U nhỏ hơn 1, gradient tắt dần theo cấp số nhân; nếu lớn hơn 1, nó bùng nổ. Với trọng số 0.9 và 100 bước, hệ số còn 0.9^100 ≈ 2.7e−5 — tín hiệu học từ đầu câu gần như bằng 0.

## 4. LSTM: cổng và đường cao tốc cộng

LSTM thêm cell state và ba cổng:

f_t = σ(W_f x_t + U_f h_{t-1} + b_f)      quên
i_t = σ(W_i x_t + U_i h_{t-1} + b_i)      ghi
o_t = σ(W_o x_t + U_o h_{t-1} + b_o)      xuất
g_t = tanh(W_g x_t + U_g h_{t-1} + b_g)   ứng viên
c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
h_t = o_t ⊙ tanh(c_t)

Điểm cốt lõi: cell state cập nhật *cộng*, không phải nhân ma trận lặp. Gradient chảy dọc theo đường cộng đó, chỉ bị nhân bởi cổng quên. Nếu f gần 1, gradient đi được hàng trăm bước. Đây là "gradient highway", không phải phép màu.

GRU rút gọn còn hai cổng (update, reset), thường tương đương về chất lượng với ít tham số hơn.

## 5. Bidirectional và pooling

RNN hai chiều chạy một chiều xuôi và một chiều ngược, ghép hidden state. Mỗi token thấy cả trái và phải — cần thiết cho gán nhãn chuỗi (POS, NER).

Pooling cho phân loại: max-pool thường thắng last-state vì thông tin ở cuối chuỗi dài dễ lấn át last state; mean-pool ổn định nhưng pha loãng tín hiệu cục bộ.

## 6. Vì sao vẫn chưa đủ

Ba vấn đề tồn tại kể cả với LSTM:

1. **Nút cổ chai tuần tự.** Chuỗi 1000 token cần 1000 bước forward/backward nối tiếp. Không song song hóa theo thời gian.
2. **Context vector cố định trong encoder–decoder.** Decoder chỉ thấy hidden state cuối của encoder, nén toàn bộ đầu vào. Đầu vào dài mất chi tiết. Bài 09 xử lý trực tiếp.
3. **Trần phụ thuộc xa.** LSTM tốt hơn RNN nhưng vẫn khó truyền thông tin cụ thể qua 200+ bước.

Attention giải cả ba. Transformer bỏ recurrence. Bài 10 là điểm chuyển.

## 7. Khi nào vẫn dùng CNN/RNN (2026)

- **Edge/on-device.** TextCNN + GloVe nhỏ hơn transformer 10–100 lần. Nếu đích là điện thoại, đây là stack.
- **Streaming/online.** RNN xử lý từng token khi dữ liệu đến; transformer cần cả chuỗi.
- **Baseline nhanh.** Train TextCNN vài phút trên CPU.
- **Sequence labeling ít dữ liệu.** BiLSTM-CRF vẫn production-grade cho 1k–10k câu gán nhãn.

Còn lại: transformer.


## A. Chạy code mẫu của bài

Cell dưới chạy `code/main.py` của bài 08: mô phỏng vanishing gradient theo độ dài chuỗi, rồi thực hiện một lượt convolution 1D thủ công trên 5 token × 3 chiều với filter rộng 2 và 3, kèm max-pool.


In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "08-cnns-rnns-for-text"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "CNNs and RNNs for Text")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: CNNs and RNNs for Text
Lệnh: phases\05-nlp-foundations-to-advanced\08-cnns-rnns-for-text\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
=== vanishing gradient sim ===
  len= 10  w=0.9  gradient attenuation = 3.49e-01
  len= 50  w=0.9  gradient attenuation = 5.15e-03
  len=100  w=0.9  gradient attenuation = 2.66e-05
  len=200  w=0.9  gradient attenuation = 7.06e-10
  a plain RNN loses 99.99% of its gradient by step 100.

=== TextCNN conceptual pass ===
  5 tokens x 3 embed_dim
  width-2 filter activations: [0.69, 0.2, 0.32, 0.33]
  width-3 filter activations: [0.94, 0.88, 0.62]
  max-pooled features:        [0.69, 0.94]


## B. Thực hành riêng — TextCNN bằng NumPy, filter như bộ dò n-gram

Ta dựng embedding 6 token × 4 chiều với các chiều mang nghĩa đọc được (`not`, `good`, `bad` là các trục riêng), rồi cài convolution 1D + ReLU + global max-pool đúng công thức.

Filter được đặt tay thay vì học, để con số đọc được:

- filter rộng 2 dò `not good`,
- filter rộng 2 dò `not bad`,
- filter rộng 3 dò `was not good`.

Trong mô hình thật, các trọng số này do lan truyền ngược học ra; cơ chế tính toán giống hệt. Bias âm buộc filter phải khớp cả hai (hoặc cả ba) token mới kích hoạt, nên filter `not bad` không bắn khi câu không có `bad`: một filter có thể khóa vào một token duy nhất nếu bias cho phép.

Sau đó ta đảo vị trí cụm `not good` lên đầu câu. Vì max-pool lấy max theo thời gian, feature map thay đổi nhưng vector pooled giữ nguyên. Đây chính là tính bất biến vị trí — hữu ích cho phân loại, bất lợi khi cần biết bằng chứng nằm ở đâu.


In [2]:
import numpy as np

TOKENS = ["the", "food", "was", "not", "good", "."]
EMBEDDINGS = np.array([
    [0.1, 0.0, 0.0, 0.1],   # the
    [0.2, 0.1, 0.0, 0.0],   # food
    [0.0, 0.0, 0.1, 0.2],   # was
    [1.0, 0.0, 0.0, 0.0],   # not
    [0.0, 1.0, 0.0, 0.0],   # good
    [0.0, 0.0, 0.0, 1.0],   # .
])

FILTERS = {
    "not_good(w=2)": (np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]), -1.0),
    "not_bad(w=2)": (np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0]]), -1.0),
    "was_not_good(w=3)": (np.array([[0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]), -1.0),
}


def conv1d(matrix, weights, bias):
    width = weights.shape[0]
    windows = len(matrix) - width + 1
    out = np.zeros(windows)
    for i in range(windows):
        out[i] = max(0.0, float(np.sum(matrix[i:i + width] * weights) + bias))
    return out


def max_pool(activations):
    return float(np.max(activations)) if len(activations) else 0.0


print("embedding matrix:", EMBEDDINGS.shape, "= (L, d)")
print(f"{'token':6s} {'d0':>5s} {'d1':>5s} {'d2':>5s} {'d3':>5s}")
for token, vector in zip(TOKENS, EMBEDDINGS):
    print(f"{token:6s} " + " ".join(f"{value:5.2f}" for value in vector))

pooled = []
for name, (weights, bias) in FILTERS.items():
    activations = conv1d(EMBEDDINGS, weights, bias)
    pooled.append(max_pool(activations))
    print(f"\nfilter {name:20s} shape={weights.shape}")
    print("  feature map (L-k+1 =", len(activations), "):", np.round(activations, 2))
    print("  max-pool:", round(pooled[-1], 2))

pooled = np.array(pooled)
print("\nvector pooled cuối:", np.round(pooled, 2), "shape =", pooled.shape)
print("Kích thước này không phụ thuộc độ dài câu; chỉ phụ thuộc số filter.")

print("\n--- bất biến vị trí ---")
order_a = ["the", "food", "was", "not", "good", "."]
order_b = ["not", "good", "the", "food", "was", "."]
index = {token: i for i, token in enumerate(TOKENS)}
weights, bias = FILTERS["not_good(w=2)"]
for label, order in (("gốc", order_a), ("đảo cụm lên đầu", order_b)):
    matrix = np.array([EMBEDDINGS[index[token]] for token in order])
    activations = conv1d(matrix, weights, bias)
    print(f"  {label:18s} activations={np.round(activations, 2)}  max={max_pool(activations):.2f}")
print("Cùng một detector, cùng max-pool, vị trí khác nhau -> cùng feature.")


embedding matrix: (6, 4) = (L, d)
token     d0    d1    d2    d3
the     0.10  0.00  0.00  0.10
food    0.20  0.10  0.00  0.00
was     0.00  0.00  0.10  0.20
not     1.00  0.00  0.00  0.00
good    0.00  1.00  0.00  0.00
.       0.00  0.00  0.00  1.00

filter not_good(w=2)        shape=(2, 4)
  feature map (L-k+1 = 5 ): [0. 0. 0. 1. 0.]
  max-pool: 1.0

filter not_bad(w=2)         shape=(2, 4)
  feature map (L-k+1 = 5 ): [0. 0. 0. 0. 0.]
  max-pool: 0.0

filter was_not_good(w=3)    shape=(3, 4)
  feature map (L-k+1 = 4 ): [0. 0. 1. 0.]
  max-pool: 1.0

vector pooled cuối: [1. 0. 1.] shape = (3,)
Kích thước này không phụ thuộc độ dài câu; chỉ phụ thuộc số filter.

--- bất biến vị trí ---
  gốc                activations=[0. 0. 0. 1. 0.]  max=1.00
  đảo cụm lên đầu    activations=[1. 0. 0. 0. 0.]  max=1.00
Cùng một detector, cùng max-pool, vị trí khác nhau -> cùng feature.


## C. Thí nghiệm — LSTM một bước, pooling và tốc độ suy giảm gradient

Ba phép đo nhỏ, mỗi phép làm rõ một khẳng định ở phần lý thuyết.

**1. Cổng LSTM.** Ta chạy forward một LSTM 4 chiều trên 6 bước với trọng số cố định, in giá trị trung bình của cổng quên/ghi/xuất, chuẩn của cell state và hidden state. Cổng quên gần 1 nghĩa là bộ nhớ được giữ; gần 0 nghĩa là bị xóa.

**2. Pooling.** Trên cùng chuỗi hidden state, so max-pool, mean-pool và last-state. Ba vector khác nhau; lựa chọn pooling là một giả định về việc tín hiệu nằm ở đâu trong câu.

**3. Gradient.** So `0.9^n` (RNN thuần) với xấp xỉ `f^n` cho đường cell state của LSTM. Với cổng quên 0.99, tín hiệu còn khoảng 0.37 sau 100 bước; với 0.9 chỉ còn 2.7e−5. Đây là toàn bộ khác biệt giữa "nhớ được" và "quên sạch".

Cuối cùng là bảng chi phí song song: CNN xử lý mọi cửa sổ cùng lúc, RNN cần n bước nối tiếp.


In [3]:
import numpy as np

rng = np.random.default_rng(11)
HIDDEN = 4
INPUT = 3


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


Wx = rng.normal(0, 0.4, size=(4 * HIDDEN, INPUT))
Wh = rng.normal(0, 0.4, size=(4 * HIDDEN, HIDDEN))
bias = np.array([1.5] * HIDDEN + [0.5] * HIDDEN + [0.0] * HIDDEN + [-0.5] * HIDDEN)  # thiên về giữ cổng quên cao

X = rng.normal(0, 0.7, size=(6, INPUT))

h = np.zeros(HIDDEN)
c = np.zeros(HIDDEN)
hidden_states, gate_log = [], []
for step, x in enumerate(X):
    z = Wx @ x + Wh @ h + bias
    forget = sigmoid(z[0:HIDDEN])
    write = sigmoid(z[HIDDEN:2 * HIDDEN])
    output = sigmoid(z[2 * HIDDEN:3 * HIDDEN])
    candidate = np.tanh(z[3 * HIDDEN:4 * HIDDEN])
    c = forget * c + write * candidate
    h = output * np.tanh(c)
    hidden_states.append(h.copy())
    gate_log.append((forget.mean(), write.mean(), output.mean(), np.linalg.norm(c), np.linalg.norm(h)))
    print(f"step {step}: f={forget.mean():.3f} i={write.mean():.3f} o={output.mean():.3f} "
          f"|c|={np.linalg.norm(c):.3f} |h|={np.linalg.norm(h):.3f}")

H = np.array(hidden_states)
print("\n--- pooling trên chuỗi hidden state ---")
print("max :", np.round(H.max(axis=0), 3))
print("mean:", np.round(H.mean(axis=0), 3))
print("last:", np.round(H[-1], 3))
print("Ba vector khác nhau: max giữ tín hiệu mạnh nhất, mean làm mượt, last chỉ nhìn cuối câu.")

print("\n--- so sánh suy giảm gradient ---")
print(f"{'bước':>5s} {'RNN 0.90^n':>14s} {'LSTM f=0.95^n':>16s} {'LSTM f=0.99^n':>16s}")
for steps in (10, 50, 100, 200):
    print(f"{steps:5d} {0.9 ** steps:14.2e} {0.95 ** steps:16.2e} {0.99 ** steps:16.2e}")
print("LSTM không 'nhớ vô hạn'; nó chỉ thay phép nhân ma trận lặp bằng cổng quên gần 1.")

print("\n--- chi phí song song ---")
print(f"{'độ dài':>8s} {'CNN: lượt tính':>16s} {'RNN: bước nối tiếp':>20s}")
for length in (100, 1000, 10000):
    print(f"{length:8d} {1:16d} {length:20d}")
print("CNN tính mọi cửa sổ độc lập; RNN buộc bước t phải chờ bước t-1.")


step 0: f=0.760 i=0.617 o=0.449 |c|=0.645 |h|=0.256
step 1: f=0.878 i=0.578 o=0.448 |c|=0.906 |h|=0.375
step 2: f=0.817 i=0.567 o=0.531 |c|=1.119 |h|=0.539
step 3: f=0.805 i=0.609 o=0.529 |c|=1.332 |h|=0.628
step 4: f=0.826 i=0.628 o=0.444 |c|=1.687 |h|=0.614
step 5: f=0.777 i=0.547 o=0.470 |c|=1.756 |h|=0.665

--- pooling trên chuỗi hidden state ---
max : [-0.024 -0.157  0.085 -0.182]
mean: [-0.246 -0.253 -0.195 -0.269]
last: [-0.267 -0.371 -0.356 -0.326]
Ba vector khác nhau: max giữ tín hiệu mạnh nhất, mean làm mượt, last chỉ nhìn cuối câu.

--- so sánh suy giảm gradient ---
 bước     RNN 0.90^n    LSTM f=0.95^n    LSTM f=0.99^n
   10       3.49e-01         5.99e-01         9.04e-01
   50       5.15e-03         7.69e-02         6.05e-01
  100       2.66e-05         5.92e-03         3.66e-01
  200       7.06e-10         3.51e-05         1.34e-01
LSTM không 'nhớ vô hạn'; nó chỉ thay phép nhân ma trận lặp bằng cổng quên gần 1.

--- chi phí song song ---
  độ dài   CNN: lượt tính   RNN: 

## D. Tự kiểm tra

1. Filter rộng 3 trong TextCNN tương đương với loại đặc trưng nào ở bài 02?
2. Vì sao max-pool giúp phân loại nhưng gây khó cho trích xuất vị trí?
3. Viết công thức một bước LSTM. Thành phần nào tạo "gradient highway"?
4. Vì sao `0.9^100` gần 0, và LSTM tránh điều đó bằng cách nào?
5. Ba hạn chế nào của RNN dẫn đến attention?
6. Nêu hai tình huống năm 2026 mà TextCNN/BiLSTM vẫn là lựa chọn đúng.

**Giới hạn:** trọng số LSTM và embedding ở đây được đặt tay/cố định để con số đọc được; đây không phải mô hình đã huấn luyện và không có số liệu chất lượng. Mục tiêu là cơ chế tính toán và trực giác định lượng, không phải benchmark.
